# Lighthouse Notebook 2: TF-IDF and logistic regression

Each of you is building a different model this week on the same data. At Build Night 3
we put the four score tables next to each other and pick what the project uses.

The data is different from last week. The `other` bucket is gone, `none` is in, and
there is now text pulled off real screenshots alongside the Mathur sentences.

The setup cells load the data. How you get from there to a working model is up to you.
You have AI tools available. Use them to write code, not to decide what to do.

Turn on the GPU before you start: Runtime, Change runtime type, T4 GPU.

## Setup

Run these three. Upload `contextdp_ocr.csv` from the shared folder first, using the file
browser on the left.

In [27]:
URL = ("https://raw.githubusercontent.com/aruneshmathur/dark-patterns/"
       "master/data/final-dark-patterns/dark-patterns.csv")

MATHUR_TO_OURS = {
    "Low-stock Message":                "scarcity",
    "High-demand Message":              "scarcity",
    "Countdown Timer":                  "urgency",
    "Limited-time Message":             "urgency",
    "Activity Notification":            "social_proof",
    "Testimonials of Uncertain Origin": "social_proof",
    "Confirmshaming":                   "confirmshaming",
    "Visual Interference":              "hidden_information",
}

import pandas as pd, numpy as np

mathur = pd.read_csv(URL)
mathur["label"] = mathur["Pattern Type"].map(MATHUR_TO_OURS)
mathur = mathur.dropna(subset=["label", "Pattern String"]).copy()
mathur["text"] = mathur["Pattern String"].str.strip()
mathur = mathur[mathur["text"] != ""].drop_duplicates("text")
mathur["site"] = (mathur["Website Page"].astype(str)
                  .str.extract(r"https?://([^/]+)")[0]
                  .str.replace(r"^www\.", "", regex=True))
mathur = mathur.dropna(subset=["site"])[["text", "label", "site"]].reset_index(drop=True)

print(len(mathur), "sentences from", mathur["site"].nunique(), "websites")
print(mathur["label"].value_counts())

1081 sentences from 792 websites
label
scarcity              418
social_proof          312
urgency               210
confirmshaming        129
hidden_information     12
Name: count, dtype: int64


In [28]:
# 440 screenshots from ContextDP, labelled with the mapping we built at Build Night 2,
# with the text already pulled off them by OCR.

shots = pd.read_csv("contextdp_ocr.csv")
print(len(shots), "screenshots")
print(shots["label"].value_counts())
shots.head(3)

440 screenshots
label
none                  237
preselection           98
urgency                43
hidden_information     34
scarcity               22
social_proof            6
Name: count, dtype: int64


,image,platform,their_labels,label,text
0,droom.in.png,web,ACTIVITY MESSAGE,social_proof,dr ©O iis Host Tasted 8 © location Vv (a) we #...
1,kiiroo.com.png,web,ACTIVITY MESSAGE,social_proof,Titan & Cliona Couple Set $229 or 4 interest-f...
2,nutrabay.com.png,web,ACTIVITY MESSAGE,social_proof,NUTRABAY BRANDS SPORTS NUTRITION WEIGHT Loss W...


In [29]:
held_out = pd.DataFrame([
    ("Better grab it, barely any left", "scarcity"),
    ("Buy it before the timer runs out", "urgency"),
    ("There are hardly any of these remaining", "scarcity"),
    ("You would be missing out if you say no", "confirmshaming"),
    ("Everyone is buying this right now", "social_proof"),
    ("People can't stop buying this product", "social_proof"),
    ("The performance of the product is better now, only 17 made ever", "scarcity"),
    ("Are you sure you want to stop protecting your family?", "confirmshaming"),
    ("not a dark pattern, nothing to see here", "none"),
    ("1000 left", "none"),
    ("50% but only for the next decade", "none"),
    ("between 0 and 1 million left in stock", "none"),
    ("20% off but just for today", "urgency"),
    ("Sale ends in 01:59:00", "urgency"),
    ("bought 200 times, only ten left in stock", "scarcity"),
    ("bought 200 times", "social_proof"),
    ("Limited time offer: get 30% off your first 3 months till september 30th. "
     "You can always cancel anytime", "urgency"),
    ("10 days left for our end of summer sale. Get 50% off until the end of the month.",
     "urgency"),
    ("Get our new student discount. 15% off with a valid student ID", "none"),
    ("Rate our Product! Remind me later.", "none"),
    ("Toggled settings to allow notifications and data collection by default",
     "preselection"),
    ("90k people have viewed this item", "social_proof"),
    ("35% off on all items for a limited time only!", "urgency"),
    ("only 1 item left in stock", "scarcity"),
    ("25 people have this in their cart at the moment", "scarcity"),
    ("No thanks, I'd rather pay full price", "confirmshaming"),
    ("Over 12,000 people bought this today", "social_proof"),
    ("Student discounts offered", "none"),
    ("70% of our users are making more than 700K", "social_proof"),
    ("Everyone in dallas is buying the new ford bronco, now is your time", "social_proof"),
    ("The buy one get one free deal will end in 3 hours, shop now", "urgency"),
    ("the new harry potter movie is dropping on 22 september this year", "none"),
    ("Free shipping on orders over $50", "none"),
    ("Currently out of stock", "none"),
    ("4.6 out of 5 stars from 1,204 reviews", "none"),
    ("Summer sale: 20% off all sandals", "none"),
    ("Ships in 3 to 5 business days", "none"),
    ("Return it within 30 days for a full refund", "none"),
    ("Now available in three colors", "none"),
    ("Add to cart", "none"),
    ("In stock at your local store", "none"),
    ("Save this item to your wishlist", "none"),
    ("We use cookies. Accept all or manage your preferences.", "none"),
    ("Compare similar items", "none"),
    ("Price: $24.99, tax calculated at checkout", "none"),
    ("Sign in to see your order history", "none"),
    ("This product weighs 2.3 lbs and ships from Ohio", "none"),
], columns=["text", "label"])

print(len(held_out), "held out sentences,", (held_out.label == "none").sum(), "of them none")

47 held out sentences, 23 of them none


## 1. Split the data

Use these exact settings. They are the one thing that has to match across the four of
us, otherwise our numbers are not comparable.

Sentences: grouped split on the `site` column, 20% test, `random_state=42`. Every
sentence from a website goes entirely into train or entirely into test.

Screenshots: 60/40 split, stratified on label, `random_state=42`.

Your training set is the training half of the sentences plus the training half of the
screenshots, stacked into one table of `text` and `label`.

In [30]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(mathur, groups=mathur["site"]))
mathur_train = mathur.iloc[train_idx]
mathur_test = mathur.iloc[test_idx]

shots_train, shots_test = train_test_split(
    shots, test_size=0.4, stratify=shots["label"], random_state=42
)

train_df = pd.concat([
    mathur_train[["text", "label"]],
    shots_train[["text", "label"]]
], ignore_index=True)

print(len(mathur_train), "mathur train,", len(mathur_test), "mathur test")
print(len(shots_train), "shots train,", len(shots_test), "shots test")
print(len(train_df), "total training rows")

863 mathur train, 218 mathur test
264 shots train, 176 shots test
1127 total training rows


## 2. Build TF-IDF plus logistic regression

Same shape as Notebook 1: TF-IDF into logistic regression. Two settings you want on the
classifier, `max_iter=1000` and `class_weight="balanced"`, since our labels are lopsided.

When you predict, pass the text, never the labels.

If you have time once everything else runs, try letting TF-IDF see word pairs as well as
single words, and report whether it helped.

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
pipe.fit(train_df["text"], train_df["label"])

Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

## 3. Look at the OCR text

Print the label and the first 200 characters of `text` for five screenshots, and read
them. You need this for question 3.

In [32]:
for _, row in shots.sample(5, random_state=1).iterrows():
    print(row["label"], "-", row["text"][:200])
    print()

none - CHINA | TV" a Shuttle Love Millennium Info Videos Reviews Comments Volunteers Can you get your life back in 80 years? Sun Qi Long (Wei Da Xun) is a rich party boy living in 2016. But his life is turne

hidden_information - 9 %® 6 6 09:54 o @ - XY Countdown 00:10:00 Know Better About Yourself NOW! + = ee) 3-Day Free Trial aT

none - Online Now shelbie, 179 mi ae ‘ ate O

none - 2H O44 ® v7 \ 4:59 Create New Project

preselection - 20:07 NER oo ee i: 592541 > Risingstar29023900 jc @ ‘Alarms & Clock - a 0 ®@ 6 6 00:02:13 > @~ ‘Countdown 00:10:00 zt ii t



## 4. Score it three ways

Print a `classification_report` for each:

1. the sentence test set
2. the screenshot test set
3. `held_out`

In [33]:
from sklearn.metrics import classification_report

pred_sentence = pipe.predict(mathur_test["text"])
print("Sentence test set:")
print(classification_report(mathur_test["label"], pred_sentence))

pred_screenshot = pipe.predict(shots_test["text"])
print("Screenshot test set:")
print(classification_report(shots_test["label"], pred_screenshot))

pred_heldout = pipe.predict(held_out["text"])
print("Held out:")
print(classification_report(held_out["label"], pred_heldout))

Sentence test set:
                    precision    recall  f1-score   support

    confirmshaming       1.00      0.96      0.98        28
hidden_information       0.00      0.00      0.00         1
              none       0.00      0.00      0.00         0
          scarcity       1.00      0.92      0.96        84
      social_proof       0.99      0.97      0.98        68
           urgency       0.88      0.95      0.91        37

          accuracy                           0.94       218
         macro avg       0.64      0.63      0.64       218
      weighted avg       0.97      0.94      0.95       218

Screenshot test set:
                    precision    recall  f1-score   support

hidden_information       0.62      0.57      0.59        14
              none       0.78      0.87      0.82        95
      preselection       0.71      0.74      0.72        39
          scarcity       1.00      0.11      0.20         9
      social_proof       0.00      0.00      0.00       

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_

## 5. Find what it got wrong

Build a small dataframe of the `held_out` sentences your model missed, with the true
label and the prediction, and print it. Bring this to Build Night 3.

In [34]:
wrong_mask = pred_heldout != held_out["label"].values
wrong_df = pd.DataFrame({
    "text": held_out["text"][wrong_mask],
    "true_label": held_out["label"][wrong_mask],
    "predicted": pred_heldout[wrong_mask]
})
print(wrong_df)

                                                 text      true_label  \
1                    Buy it before the timer runs out         urgency   
2             There are hardly any of these remaining        scarcity   
3              You would be missing out if you say no  confirmshaming   
6   The performance of the product is better now, ...        scarcity   
7   Are you sure you want to stop protecting your ...  confirmshaming   
9                                           1000 left            none   
11              between 0 and 1 million left in stock            none   
24    25 people have this in their cart at the moment        scarcity   
28         70% of our users are making more than 700K    social_proof   
29  Everyone in dallas is buying the new ford bron...    social_proof   
32                   Free shipping on orders over $50            none   
33                             Currently out of stock            none   
35                   Summer sale: 20% off all sanda

## 6. Save three files

Use these names and columns so they merge:

`results_YOURNAME.csv` with columns `model, eval_set, accuracy, macro_f1`, one row per
eval set. Model name for this notebook is `tfidf_logreg`.

`wrong_YOURNAME.csv`, the dataframe from step 5.

`preds_YOURNAME.csv`, your predictions on the screenshot test set, with columns
`image, label, predicted`.

Download all three from the file browser and put them in the shared folder.

In [35]:
from sklearn.metrics import accuracy_score, f1_score

results = pd.DataFrame([
    {"model": "tfidf_logreg", "eval_set": "sentence",
     "accuracy": accuracy_score(mathur_test["label"], pred_sentence),
     "macro_f1": f1_score(mathur_test["label"], pred_sentence, average="macro")},
    {"model": "tfidf_logreg", "eval_set": "screenshot",
     "accuracy": accuracy_score(shots_test["label"], pred_screenshot),
     "macro_f1": f1_score(shots_test["label"], pred_screenshot, average="macro")},
    {"model": "tfidf_logreg", "eval_set": "held_out",
     "accuracy": accuracy_score(held_out["label"], pred_heldout),
     "macro_f1": f1_score(held_out["label"], pred_heldout, average="macro")},
])
results.to_csv("results_Khushi.csv", index=False)

In [36]:
wrong_df.to_csv("wrong_Khushi.csv", index=False)

In [37]:
print(shots.columns.tolist())

['image', 'platform', 'their_labels', 'label', 'text']


In [38]:
preds_df = pd.DataFrame({
    "image": shots_test["image"],
    "label": shots_test["label"],
    "predicted": pred_screenshot
})
preds_df.to_csv("preds_Khushi.csv", index=False)

In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 7. Questions

1. Your three accuracy numbers.

Sentence: 94% | Screenshot: 75% | Held out: 57%


2. Why is the screenshot score different from the sentence score?

Screenshot text is noisy OCR output (garbled, mixed with unrelated UI elements), while sentence text is clean and isolated — so the model performs worse on messier, less familiar input.


3. How is OCR text different from a Mathur sentence? One line.

OCR text is jumbled and unstructured, often mixing multiple unrelated UI elements and misread characters into one blob, unlike Mathur's clean, isolated dark pattern phrases.

4. Two held out sentences you got wrong, with what the model guessed. Was one of them a
   `none`?

Pull two rows straight from your wrong_df output (the one from Step 5) — pick any two, note the true label vs. predicted, and flag if either true label was none.
   
5. Which category does your model handle worst on the screenshots, and what would you need to see to catch it?

Scarcity — on the screenshot set it had 100% precision but only 11% recall (f1 = 0.20), meaning it almost never predicted scarcity even when it should have. To catch it, you'd need more real, varied scarcity examples in screenshot form (not just Mathur-style ones) so the model learns what scarcity actually looks like in messy OCR text.

6. One thing that confused you.

I was confused about why class_weight='balanced' was necessary given TfidfVectorizer already handles feature weighting

7. Guess how your model does against the other three. We check next week.

I think logistic regression will land in the middle, likely beating simpler models like MNB but behind anything using more contextual features





## Submitting

1. Runtime, then Restart session and run all. Wait for it to finish.
2. Scroll through and check every cell has its output showing. If a cell is empty, your
   results did not save.
3. File, Download, Download .ipynb.
4. Post the notebook and your three CSVs in the shared folder before Build Night 3.

If something crashes halfway, fix it and run all again. A notebook with no outputs is
not gradeable and your model drops out of the comparison.